In [2]:
from torchvision.models import efficientnet_v2_s, efficientnet_v2_m, efficientnet_v2_l
from torchvision.models import EfficientNet_V2_S_Weights, EfficientNet_V2_M_Weights, EfficientNet_V2_L_Weights
from torch import nn

In [11]:
class Efficientnet_small(nn.Module):
    def __init__(self):
        super(Efficientnet_small, self).__init__()
        self.efficientnet = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

    def forward(self, x):
        return self.efficientnet(x)



In [12]:
test = Efficientnet_small()

In [23]:
test.efficientnet.features[3]

Sequential(
  (0): FusedMBConv(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(48, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (stochastic_depth): StochasticDepth(p=0.030000000000000006, mode=row)
  )
  (1): FusedMBConv(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(256, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(256, 64, kernel_size=(1, 1), stri

In [24]:
import torch

class FpnCombine(nn.Module):
    def __init__(self, num_inputs, epsilon=1e-4):
        super(FpnCombine, self).__init__()
        # Initialize learnable scalar weights with ones
        self.edge_weights = nn.Parameter(torch.ones(num_inputs, dtype=torch.float32))
        self.epsilon = epsilon

    def forward(self, inputs):
        """
        inputs: A list of resampled tensors of identical shape
        """
        # Apply ReLU to ensure all weights are strictly positive
        weights = torch.relu(self.edge_weights)

        # Calculate the normalization denominator
        weights_sum = torch.sum(weights) + self.epsilon

        # Perform Fast Normalized Fusion
        # Multiply each input tensor by its normalized weight
        out = torch.stack(
            [(inputs[i] * weights[i]) / weights_sum for i in range(len(inputs))],
            dim=-1
        )

        # Sum across the stacked dimension to produce the final fused tensor
        return torch.sum(out, dim=-1)

In [25]:
class SeparableConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SeparableConvBlock, self).__init__()
        self.depthwise_conv = nn.Conv2d(
            in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False
        )
        self.pointwise_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)
        self.bn = nn.BatchNorm2d(out_channels, momentum=0.01, eps=1e-3)
        self.swish = nn.SiLU()

    def forward(self, x):
        x = self.depthwise_conv(x)
        x = self.pointwise_conv(x)
        x = self.bn(x)
        return self.swish(x)

class HeadNet(nn.Module):
    def __init__(self, in_channels, num_layers, num_outputs):
        super(HeadNet, self).__init__()
        self.convs = nn.ModuleList()
        # Final prediction convolution
        self.predict = nn.Conv2d(in_channels, num_outputs, kernel_size=3, padding=1, bias=True)

    def forward(self, x):
        for conv in self.convs:
            x = conv(x)
        return self.predict(x)

In [26]:
import math

def initialize_head_biases(model, pi=0.01):
    bias_value = -math.log((1 - pi) / pi)
    for name, module in model.named_modules():
        if 'class_net.predict' in name and isinstance(module, nn.Conv2d):
            # Fill the bias tensor with the calculated prior
            module.bias.data.fill_(bias_value)
            # Initialize weights with a standard normal distribution
            fan_out = module.kernel_size * module.kernel_size * module.out_channels
            module.weight.data.normal_(0, math.sqrt(2.0 / fan_out))

In [27]:
import numpy as np

def generate_anchors(base_size, ratios, scales):
    """
    Generates 9 anchor box geometries relative to a center point.
    """
    num_anchors = len(ratios) * len(scales)
    anchors = np.zeros((num_anchors, 4))

    # Expand scales and ratios to create 9 distinct combinations
    scales_grid, ratios_grid = np.meshgrid(scales, ratios)
    scales_flat = scales_grid.flatten()
    ratios_flat = ratios_grid.flatten()

    # Calculate width and height based on the base size, scale, and ratio
    # area = base_size^2 * scale^2.  w * h = area,  h / w = ratio
    widths = base_size * scales_flat * np.sqrt(1.0 / ratios_flat)
    heights = base_size * scales_flat * np.sqrt(ratios_flat)

    # Format anchors as [-w/2, -h/2, w/2, h/2] to center them around (0,0)
    anchors[:, 0] = -widths / 2.0
    anchors[:, 1] = -heights / 2.0
    anchors[:, 2] = widths / 2.0
    anchors[:, 3] = heights / 2.0

    return anchors

In [28]:
import torch

def decode_predictions(predictions, anchors, variances=[0.1, 0.2]):
    """
    predictions: Tensor of shape containing (dx, dy, dw, dh)
    anchors: Tensor of shape [Num_Anchors, 4] containing center coordinates (px, py, pw, ph)
    """
    # Decode center coordinates (x, y)
    b_x_y = anchors[:, :2] + predictions[:, :, :2] * anchors[:, 2:] * variances

    # Decode dimensions (width, height) using exponential mapping
    b_w_h = anchors[:, 2:] * torch.exp(predictions[:, :, 2:] * variances)

    # Combine decoded center coordinates and dimensions
    decoded_boxes_center = torch.cat([b_x_y, b_w_h], dim=-1)

    return decoded_boxes_center